# 04 KIỂM THỬ BỘ QUY TẮC & HÌNH HỌC LUẬT CỜ (RULES TESTING)

Tài liệu này kiểm tra chi tiết module `commander_game_engine.rules`:
1. **Hình học di chuyển (`movement_shape`)**: Tầm đi, hướng đi và quy tắc ăn quân của 11 loại quân.
2. **Địa hình & Vượt sông (`_terrain_ok`)**: Biển, sông sâu, ngầm lội (Ford) và phân định quân vượt sông.
3. **Vật cản trên hành trình (`path_points` & vật cản)**: Bị chặn đường đi thường vs Pháo bắn xuyên vật cản.
4. **Quy tắc cấm Tư lệnh đối mặt (`is_commanders_facing`)**: Ghim quân cản, cấm tự sát, cấm lộ mặt Tướng.
5. **Xác thực hành động toàn diện (`validate_action`)**: MOVE, CAPTURE, BOMBARD.

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Cấu hình nạp module
cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / "packages").is_dir() and (p / "research").is_dir()), None)
if str(PROJECT_ROOT / "packages") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "packages"))

from commander_game_engine.board import Board, Terrain, EdgeType, Position
from commander_game_engine.game_state import GameState, Piece, PieceType, Player
from commander_game_engine.actions import ActionType, AtomicAction
from commander_game_engine import rules
from commander_game_engine.rules import RuleCode, RuleResult

print("Nạp module packages/commander_game_engine/rules thành công!")

Nạp module packages/commander_game_engine/rules thành công!


## 1. Kiểm tra hình học & tầm đi cơ bản (`movement_shape`)

In [2]:
# 1. Xe tăng: 4 hướng thẳng 1-2 đoạn
tank = Piece(id="red_tank", piece_type=PieceType.TANK, owner=Player.RED, position=(3, 3))
assert rules.movement_shape(tank, (3, 5)).valid == True   # Đi thẳng 2 đoạn -> Hợp lệ
assert rules.movement_shape(tank, (3, 4)).valid == True   # Đi thẳng 1 đoạn -> Hợp lệ
assert rules.movement_shape(tank, (3, 6)).code == RuleCode.OUT_OF_RANGE  # 3 đoạn -> Quá tầm
assert rules.movement_shape(tank, (5, 5)).code == RuleCode.OUT_OF_RANGE  # Đi chéo -> Cấm

# 2. Dân quân: 8 hướng 1 đoạn
militia = Piece(id="red_militia", piece_type=PieceType.MILITIA, owner=Player.RED, position=(4, 4))
assert rules.movement_shape(militia, (5, 5)).valid == True  # Chéo 1 đoạn -> Hợp lệ
assert rules.movement_shape(militia, (4, 5)).valid == True  # Thẳng 1 đoạn -> Hợp lệ
assert rules.movement_shape(militia, (6, 6)).code == RuleCode.OUT_OF_RANGE  # 2 đoạn -> Quá tầm

# 3. Tư lệnh: Đi thẳng không giới hạn, ăn quân 1 đoạn
cmd = Piece(id="red_cmd", piece_type=PieceType.COMMANDER, owner=Player.RED, position=(0, 5))
assert rules.movement_shape(cmd, (10, 5), capture=False).valid == True  # Đi xa 10 đoạn -> Hợp lệ
assert rules.movement_shape(cmd, (1, 5), capture=True).valid == True    # Ăn 1 đoạn -> Hợp lệ
assert rules.movement_shape(cmd, (2, 5), capture=True).code == RuleCode.OUT_OF_RANGE  # Ăn 2 đoạn -> Quá tầm

# 4. Pháo binh: 8 hướng 1-3 đoạn
artillery = Piece(id="red_artillery", piece_type=PieceType.ARTILLERY, owner=Player.RED, position=(2, 3))
assert rules.movement_shape(artillery, (5, 3)).valid == True   # 3 đoạn thẳng -> Hợp lệ
assert rules.movement_shape(artillery, (5, 6)).valid == True   # 3 đoạn chéo -> Hợp lệ
assert rules.movement_shape(artillery, (6, 3)).code == RuleCode.OUT_OF_RANGE # 4 đoạn -> Quá tầm

# 5. Sở chỉ huy: Bất động
hq = Piece(id="red_hq", piece_type=PieceType.HEADQUARTERS, owner=Player.RED, position=(0, 5))
assert rules.movement_shape(hq, (0, 6)).code == RuleCode.OUT_OF_RANGE

print("PASS: Tất cả các bài kiểm tra movement_shape cơ bản đều chính xác!")

PASS: Tất cả các bài kiểm tra movement_shape cơ bản đều chính xác!


## 2. Kiểm tra tương tác địa hình & Vượt sông (`_terrain_ok`)

In [3]:
state = GameState()

# A. Quân bộ đi vào biển
inf = Piece(id="red_inf", piece_type=PieceType.INFANTRY, owner=Player.RED, position=(3, 2))
res_sea = rules._terrain_ok(state, inf, (3, 2), (3, 1), capture=False)
assert res_sea.code == RuleCode.TERRAIN_BLOCKED
print("PASS: Quân bộ đi vào biển (x=1) -> Bị từ chối TERRAIN_BLOCKED.")

# B. Hải quân đi vào đất liền sâu (x >= 3)
navy = Piece(id="red_navy", piece_type=PieceType.NAVY, owner=Player.RED, position=(3, 1))
res_land = rules._terrain_ok(state, navy, (3, 1), (3, 3), capture=False)
assert res_land.code == RuleCode.TERRAIN_BLOCKED
print("PASS: Hải quân đi vào đất liền (x=3) -> Bị từ chối TERRAIN_BLOCKED.")

# C. Vượt sông sâu giữa hàng 5 và 6
# 1. Xe tăng thuộc _DEEP_RIVER_CROSSERS -> Qua được sông sâu
res_tank_river = rules._terrain_ok(state, tank, (5, 3), (6, 3), capture=False)
assert res_tank_river.valid == True
print("PASS: Xe tăng (loại vượt sông sâu) bơi qua sông sâu (x=3) -> Hợp lệ.")

# 2. Pháo binh (quân nặng không thuộc _DEEP_RIVER_CROSSERS):
# Đi thường qua nước sâu -> Bị chặn
res_art_deep = rules._terrain_ok(state, artillery, (5, 3), (6, 3), capture=False)
assert res_art_deep.code == RuleCode.TERRAIN_BLOCKED
print("PASS: Pháo binh đi thường qua nước sâu (x=3) -> Bị từ chối TERRAIN_BLOCKED.")

# Đi qua ngầm (Ford tại x=5) -> Hợp lệ
res_art_ford = rules._terrain_ok(state, artillery, (5, 5), (6, 5), capture=False)
assert res_art_ford.valid == True
print("PASS: Pháo binh đi qua lối ngầm (x=5) -> Hợp lệ.")

# Ăn quân thế chỗ ở bờ đối diện -> Hợp lệ
res_art_cap = rules._terrain_ok(state, artillery, (5, 3), (6, 3), capture=True)
assert res_art_cap.valid == True
print("PASS: Pháo binh ăn quân thế chỗ qua sông sâu -> Hợp lệ.")

PASS: Quân bộ đi vào biển (x=1) -> Bị từ chối TERRAIN_BLOCKED.
PASS: Hải quân đi vào đất liền (x=3) -> Bị từ chối TERRAIN_BLOCKED.
PASS: Xe tăng (loại vượt sông sâu) bơi qua sông sâu (x=3) -> Hợp lệ.
PASS: Pháo binh đi thường qua nước sâu (x=3) -> Bị từ chối TERRAIN_BLOCKED.
PASS: Pháo binh đi qua lối ngầm (x=5) -> Hợp lệ.
PASS: Pháo binh ăn quân thế chỗ qua sông sâu -> Hợp lệ.


## 3. Kiểm tra vật cản & Đặc quyền bắn xuyên của Pháo binh

In [4]:
# Thiết lập bàn cờ có quân cản
st_obs = GameState()
tank_mover = Piece(id="red_tank", piece_type=PieceType.TANK, owner=Player.RED, position=(3, 3))
blocker = Piece(id="red_inf", piece_type=PieceType.INFANTRY, owner=Player.RED, position=(3, 4))
st_obs.add_piece(tank_mover)
st_obs.add_piece(blocker)

# Xe tăng đi 2 đoạn bị cản ở đoạn 1 (3, 4)
act_tank = AtomicAction(ActionType.MOVE, "red_tank", (3, 3), (3, 5))
verdict_tank = rules.validate_action(st_obs, act_tank)
assert verdict_tank.code == RuleCode.PATH_BLOCKED
print("PASS: Xe tăng bị cản ở đoạn giữa -> PATH_BLOCKED.")

# Pháo binh đi thường bị cản -> PATH_BLOCKED
st_art = GameState()
art_piece = Piece(id="red_art", piece_type=PieceType.ARTILLERY, owner=Player.RED, position=(3, 2))
art_blocker = Piece(id="red_blocker", piece_type=PieceType.INFANTRY, owner=Player.RED, position=(3, 3))
enemy_target = Piece(id="blue_inf", piece_type=PieceType.INFANTRY, owner=Player.BLUE, position=(3, 5))
st_art.add_piece(art_piece)
st_art.add_piece(art_blocker) # Cản ở (3, 3)
st_art.add_piece(enemy_target)

act_art_move = AtomicAction(ActionType.MOVE, "red_art", (3, 2), (3, 4))
verdict_art_move = rules.validate_action(st_art, act_art_move)
assert verdict_art_move.code == RuleCode.PATH_BLOCKED
print("PASS: Pháo binh đi thường bị quân cản chặn -> PATH_BLOCKED.")

# Pháo binh BẮN ĂN QUÂN xuyên qua quân cản -> HỢP LỆ!
act_art_cap = AtomicAction(ActionType.CAPTURE, "red_art", (3, 2), (3, 5), target_piece_id="blue_inf")
verdict_art_cap = rules.validate_action(st_art, act_art_cap)
assert verdict_art_cap.valid == True
print("PASS: Pháo binh bắn ăn quân địch xuyên qua quân cản ở giữa -> Chấp nhận thành công (RuleCode.OK)!")

PASS: Xe tăng bị cản ở đoạn giữa -> PATH_BLOCKED.
PASS: Pháo binh đi thường bị quân cản chặn -> PATH_BLOCKED.
PASS: Pháo binh bắn ăn quân địch xuyên qua quân cản ở giữa -> Chấp nhận thành công (RuleCode.OK)!


## 4. Kiểm tra quy tắc cấm Tư lệnh đối mặt nhau (`commanders_face`)

In [5]:
# Thế cờ: Hai Tư lệnh trên cùng cột 5
st_face = GameState()
cmd_red = Piece(id="red_cmd", piece_type=PieceType.COMMANDER, owner=Player.RED, position=(0, 5))
cmd_blue = Piece(id="blue_cmd", piece_type=PieceType.COMMANDER, owner=Player.BLUE, position=(11, 5))
shield_inf = Piece(id="red_shield", piece_type=PieceType.INFANTRY, owner=Player.RED, position=(5, 5))

st_face.add_piece(cmd_red)
st_face.add_piece(cmd_blue)
st_face.add_piece(shield_inf)

# 1. Khi có quân che chắn ở (5, 5) -> Không bị đối mặt
assert rules.commanders_face(rules._visible_occupancy(st_face)) == False
assert rules.validate_commander_exposure(st_face).valid == True
print("PASS: Có quân cản ở giữa -> commanders_face == False & validate_commander_exposure OK.")

# 2. Quân che chắn duy nhất di chuyển rời đi -> Bị cấm COMMANDERS_FACING!
act_move_shield = AtomicAction(ActionType.MOVE, "red_shield", (5, 5), (5, 4))
verdict_shield = rules.validate_action(st_face, act_move_shield)
assert verdict_shield.code == RuleCode.COMMANDERS_FACING
print("PASS: Quân cản duy nhất di chuyển làm lộ mặt Tướng -> Bị từ chối COMMANDERS_FACING.")

# 3. Tư lệnh tự đi vào đường trống đối mặt Tư lệnh đối phương -> Bị cấm!
st_face_2 = GameState()
st_face_2.add_piece(cmd_red)
cmd_blue_col4 = Piece(id="blue_cmd", piece_type=PieceType.COMMANDER, owner=Player.BLUE, position=(11, 4))
st_face_2.add_piece(cmd_blue_col4)

act_cmd_suicide = AtomicAction(ActionType.MOVE, "red_cmd", (0, 5), (0, 4))
verdict_cmd = rules.validate_action(st_face_2, act_cmd_suicide)
assert verdict_cmd.code == RuleCode.COMMANDERS_FACING
print("PASS: Tư lệnh tự đi vào đường trống đối mặt trực diện -> Bị từ chối COMMANDERS_FACING.")

PASS: Có quân cản ở giữa -> commanders_face == False & validate_commander_exposure OK.
PASS: Quân cản duy nhất di chuyển làm lộ mặt Tướng -> Bị từ chối COMMANDERS_FACING.
PASS: Tư lệnh tự đi vào đường trống đối mặt trực diện -> Bị từ chối COMMANDERS_FACING.


## 5. Kết luận kiểm thử Rules
Toàn bộ các quy tắc về hình học, địa hình vượt sông, bắn xuyên vật cản và cấm đối mặt Tư lệnh đã được kiểm định đạt 100% độ chính xác theo tài liệu đặc tả cờ.